# 16_E8 FINAL LABELS FIXED 2 - Al-Kafri final GT masks

Usa los labels finales `05_Final_Ground_Truth_Data/Label_Images/C1_XXXX_DY.png` y el pairing oficial HEADER_NONE ya generado. No entrena modelos; genera overlays y CSV de revisión.

In [ ]:

import importlib.util, subprocess, sys
from pathlib import Path

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name])
ensure_package('pydicom')
ensure_package('skimage', 'scikit-image')

import json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom
from PIL import Image
from scipy import ndimage
from skimage.morphology import remove_small_objects, binary_closing, disk
from skimage.transform import resize
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 160)
pd.set_option('display.max_colwidth', 180)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    print('Drive no montado automáticamente:', repr(exc))

PFI_ROOT = Path('/content/drive/MyDrive/PFI_MVP')
E7_ROOT = PFI_ROOT / 'results' / 'E7_alkafri_axial_curated_subset'
E8_ROOT = PFI_ROOT / 'results' / 'E8_alkafri_official_pairing'
FIGURES_ROOT = PFI_ROOT / 'figures'
DOCS_ROOT = PFI_ROOT / 'docs'
for p in [E8_ROOT, FIGURES_ROOT, DOCS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)
OFFICIAL_FILTERED_PATH = E8_ROOT / 'E8_official_pairing_candidates_HEADER_NONE_FILTERED.csv'
GT_INDEX_PATH = E7_ROOT / 'E7_alkafri_gt_case_index.csv'
print('OFFICIAL_FILTERED_PATH:', OFFICIAL_FILTERED_PATH, OFFICIAL_FILTERED_PATH.exists())
print('GT_INDEX_PATH:', GT_INDEX_PATH, GT_INDEX_PATH.exists())
if not OFFICIAL_FILTERED_PATH.exists():
    raise FileNotFoundError('Falta E8_official_pairing_candidates_HEADER_NONE_FILTERED.csv')
if not GT_INDEX_PATH.exists():
    raise FileNotFoundError('Falta E7_alkafri_gt_case_index.csv')

def norm_case(x):
    if pd.isna(x):
        return None
    m = re.search(r'(\d{1,4})', str(x))
    return f'{int(m.group(1)):04d}' if m else None

def parse_final_label_name(path):
    name = Path(str(path)).name
    m = re.search(r'([A-Za-z]+\d*)[_-](\d{1,4})[_-]D([345])', name)
    if not m:
        return None, None, None
    return m.group(1), f'{int(m.group(2)):04d}', int(m.group(3))

def read_dicom(path):
    ds = pydicom.dcmread(str(path), force=True)
    _ = ds.pixel_array
    return ds

def normalize_image(arr):
    arr = arr.astype(np.float32)
    p1, p99 = np.percentile(arr, [1, 99])
    if p99 <= p1:
        return np.zeros_like(arr, dtype=np.float32)
    return (np.clip(arr, p1, p99) - p1) / (p99 - p1 + 1e-8)

def read_png(path):
    with Image.open(path) as img:
        arr = np.asarray(img)
    if arr.ndim == 3 and arr.shape[-1] == 4:
        arr = arr[..., :3]
    return arr

def dominant_border_value(label_arr):
    if label_arr.ndim == 2:
        border = np.concatenate([label_arr[0,:], label_arr[-1,:], label_arr[:,0], label_arr[:,-1]])
        vals, counts = np.unique(border.reshape(-1), return_counts=True)
        return vals[np.argmax(counts)]
    flat = np.concatenate([label_arr[0,:,:], label_arr[-1,:,:], label_arr[:,0,:], label_arr[:,-1,:]], axis=0)
    vals, counts = np.unique(flat.reshape(-1, flat.shape[-1]), axis=0, return_counts=True)
    return tuple(vals[np.argmax(counts)].tolist())

def build_label_map_from_final(raw):
    h, w = raw.shape[:2]
    if raw.ndim == 2:
        arr = raw.astype(np.int32)
        bg = dominant_border_value(arr)
        vals = [int(v) for v in np.unique(arr) if int(v) != int(bg)]
        label_map = np.zeros((h, w), dtype=np.uint8)
        table = []
        for class_id, v in enumerate(vals, start=1):
            m = arr == v
            label_map[m] = class_id
            table.append({'class_id': class_id, 'original_value': int(v), 'ratio': float(m.mean()), 'pixels': int(m.sum())})
        return label_map, {'mode': 'grayscale_values', 'background': int(bg), 'unique_count': int(len(vals) + 1), 'classes': table}
    arr = raw[..., :3].astype(np.uint8)
    bg = np.array(dominant_border_value(arr), dtype=np.uint8)
    flat = arr.reshape(-1, 3)
    vals, counts = np.unique(flat, axis=0, return_counts=True)
    order = np.argsort(counts)[::-1]
    label_map = np.zeros((h, w), dtype=np.uint8)
    table = []
    class_id = 1
    for idx in order:
        color = vals[idx]
        if np.all(color == bg):
            continue
        m = np.all(arr == color, axis=-1)
        label_map[m] = class_id
        table.append({'class_id': class_id, 'original_value': tuple(map(int, color.tolist())), 'ratio': float(m.mean()), 'pixels': int(m.sum())})
        class_id += 1
    return label_map, {'mode': 'rgb_colors', 'background': tuple(map(int, bg.tolist())), 'unique_count': int(len(vals)), 'classes': table}

def clean_class_mask(mask, min_size=30, close_radius=1):
    mask = mask.astype(bool)
    if mask.any() and close_radius:
        mask = binary_closing(mask, disk(close_radius))
    if mask.any():
        mask = remove_small_objects(mask, min_size=min_size)
    return mask.astype(np.uint8)

def label_map_to_clean_binary(label_map):
    out = np.zeros_like(label_map, dtype=np.uint8)
    for c in sorted([int(x) for x in np.unique(label_map) if int(x) != 0]):
        out = np.maximum(out, clean_class_mask(label_map == c))
    return out

def resize_label_map_if_needed(label_map, target_shape):
    if tuple(label_map.shape) == tuple(target_shape):
        return label_map.astype(np.uint8)
    return resize(label_map.astype(np.float32), target_shape, order=0, preserve_range=True, anti_aliasing=False).astype(np.uint8)

def component_count(mask):
    _, n = ndimage.label(mask.astype(bool))
    return int(n)

# Carga de candidatos oficiales y GT final
official_df = pd.read_csv(OFFICIAL_FILTERED_PATH)
official_df['case_id_norm'] = official_df['case_id'].apply(norm_case)
official_df['disc_id'] = pd.to_numeric(official_df['disc_id'], errors='coerce').astype('Int64')
raw_gt = pd.read_csv(GT_INDEX_PATH, low_memory=False)
raw_gt['path_lower'] = raw_gt['gt_file_path'].astype(str).str.lower()
final_png = raw_gt[
    raw_gt['extension'].astype(str).str.lower().eq('.png')
    & raw_gt['path_lower'].str.contains('05_final_ground_truth_data', na=False)
].copy()
final_png['folder_kind'] = final_png['path_lower'].apply(lambda s: 'label_images' if 'label_images' in s and 'resized' not in s else ('resized_label_images' if 'resized_label_images' in s else ('composite_images' if 'composite_images' in s and 'resized' not in s else 'resized_composite_images')))
parsed = final_png['gt_file_path'].apply(parse_final_label_name)
final_png['final_prefix'] = parsed.apply(lambda x: x[0])
final_png['case_id_norm'] = parsed.apply(lambda x: x[1])
final_png['disc_id'] = pd.to_numeric(parsed.apply(lambda x: x[2]), errors='coerce').astype('Int64')
final_png['parse_ok'] = final_png['case_id_norm'].notna() & final_png['disc_id'].notna()
print('official_df:', official_df.shape)
print('final_png:', final_png.shape, 'parse_ok:', int(final_png['parse_ok'].sum()))
display(final_png.groupby(['folder_kind', 'final_prefix'], dropna=False).size().reset_index(name='n'))

# Priorizar Label_Images reales sobre composite/resized.
priority = {'label_images': 0, 'resized_label_images': 1, 'composite_images': 2, 'resized_composite_images': 3}
final_labels = final_png[final_png['parse_ok']].copy()
final_labels['folder_priority'] = final_labels['folder_kind'].map(priority).fillna(9).astype(int)
final_labels = final_labels.sort_values(['folder_priority', 'final_prefix', 'case_id_norm', 'disc_id', 'gt_file_path'])
best_final_labels = final_labels.drop_duplicates(['case_id_norm', 'disc_id'], keep='first').copy()
print('best_final_labels:', best_final_labels.shape)
display(best_final_labels.groupby(['folder_kind', 'final_prefix'], dropna=False).size().reset_index(name='n'))

final_candidates_df = official_df.merge(
    best_final_labels[['case_id_norm','disc_id','gt_file_path','gt_relative_path','folder_kind','final_prefix']],
    on=['case_id_norm','disc_id'],
    how='inner'
).rename(columns={'gt_file_path':'final_label_file_path', 'gt_relative_path':'final_label_relative_path'})
final_candidates_df.to_csv(E8_ROOT / 'E8_FINAL_LABELS_FIXED_2_candidates.csv', index=False)
print('final_candidates_df:', final_candidates_df.shape)
display(final_candidates_df.groupby(['modality','folder_kind','final_prefix'], dropna=False).size().reset_index(name='n'))

# Visualización
VIS_N = 120
vis_df = (final_candidates_df.sort_values(['case_id_norm','modality','disc_id','instance_number'])
          .groupby(['modality','disc_id'], group_keys=False).head(25).head(VIS_N).copy())
print('Candidatos FINAL LABELS fixed_2 a visualizar:', len(vis_df))
review_rows = []
for i, (_, r) in enumerate(tqdm(vis_df.iterrows(), total=len(vis_df), desc='final label overlays fixed_2'), start=1):
    cid = f'final_labels_fixed_2_{i:03d}'
    try:
        ds = read_dicom(r['image_file_path'])
        img = ds.pixel_array.astype(np.float32)
        img_norm = normalize_image(img)
        raw_label = read_png(r['final_label_file_path'])
        label_map, info = build_label_map_from_final(raw_label)
        label_map = resize_label_map_if_needed(label_map, img.shape)
        binary = label_map_to_clean_binary(label_map)
        classes = sorted([int(x) for x in np.unique(label_map) if int(x) != 0])
        n_class_panels = min(len(classes), 4)
        fig, axes = plt.subplots(1, 4 + n_class_panels, figsize=(4 * (4 + n_class_panels), 4))
        axes[0].imshow(img_norm, cmap='gray'); axes[0].set_title('Axial IMA'); axes[0].axis('off')
        axes[1].imshow(raw_label if raw_label.ndim == 3 else raw_label, cmap=None if raw_label.ndim == 3 else 'nipy_spectral'); axes[1].set_title('Final label raw'); axes[1].axis('off')
        axes[2].imshow(label_map, cmap='nipy_spectral', vmin=0); axes[2].set_title(f'Label map\nclasses={len(classes)}'); axes[2].axis('off')
        axes[3].imshow(img_norm, cmap='gray'); axes[3].imshow(np.ma.masked_where(binary <= 0, binary), alpha=0.45, cmap='autumn'); axes[3].set_title(f'Overlay all\nratio={binary.mean():.3f} comps={component_count(binary)}'); axes[3].axis('off')
        for ax, c in zip(axes[4:], classes[:n_class_panels]):
            m = clean_class_mask(label_map == c)
            ax.imshow(img_norm, cmap='gray'); ax.imshow(np.ma.masked_where(m <= 0, m), alpha=0.55, cmap='autumn'); ax.set_title(f'class {c}\nratio={m.mean():.3f} comps={component_count(m)}'); ax.axis('off')
        fig.suptitle(f'{cid} | FINAL LABEL | case={r["case_id_norm"]} | {r["modality"]} | D={r["disc_id"]} | inst={r.get("instance_number", "?")}', fontsize=10)
        fig.tight_layout()
        fig_path = FIGURES_ROOT / f'E8_alkafri_FINAL_LABELS_FIXED_2_{i:03d}.png'
        fig.savefig(fig_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        review_rows.append({'candidate_id': cid, 'figure_path': str(fig_path), 'case_id': r['case_id_norm'], 'modality': r['modality'], 'disc_id': int(r['disc_id']), 'instance_number': r.get('instance_number'), 'final_label_source': r['folder_kind'], 'final_prefix': r['final_prefix'], 'image_file_path': r['image_file_path'], 'final_label_file_path': r['final_label_file_path'], 'label_mode': info['mode'], 'raw_unique_count': int(info['unique_count']), 'n_classes_non_bg': int(len(classes)), 'clean_foreground_ratio': float(binary.mean()), 'clean_component_count': component_count(binary), 'decode_info': json.dumps(info, ensure_ascii=False), 'auto_status': 'review', 'manual_accept': '', 'manual_reject_reason': '', 'notes': ''})
    except Exception as exc:
        review_rows.append({'candidate_id': cid, 'auto_status': 'error', 'error': repr(exc), 'case_id': r.get('case_id_norm'), 'modality': r.get('modality'), 'disc_id': r.get('disc_id')})
review_df = pd.DataFrame(review_rows)
review_path = E8_ROOT / 'E8_FINAL_LABELS_FIXED_2_visual_review_sheet.csv'
review_df.to_csv(review_path, index=False)
print('Figuras FINAL LABELS FIXED_2 generadas:', int(review_df.get('figure_path', pd.Series(dtype=object)).notna().sum()))
print('Review sheet:', review_path)
display(review_df.head(30))
display(review_df[['clean_foreground_ratio','clean_component_count','n_classes_non_bg']].describe())

report = {'notebook': '16_E8_alkafri_FINAL_LABELS_FIXED_2', 'goal': 'use 05_Final_Ground_Truth_Data/Label_Images with C1_XXXX_DY parser', 'official_candidates_loaded': int(len(official_df)), 'final_png_found': int(len(final_png)), 'final_png_parse_ok': int(final_png['parse_ok'].sum()), 'best_final_labels_case_disc': int(len(best_final_labels)), 'final_candidates_official_joined': int(len(final_candidates_df)), 'visual_review_figures': int(review_df.get('figure_path', pd.Series(dtype=object)).notna().sum()), 'review_sheet': str(review_path), 'candidate_csv': str(E8_ROOT / 'E8_FINAL_LABELS_FIXED_2_candidates.csv'), 'figures_prefix': str(FIGURES_ROOT / 'E8_alkafri_FINAL_LABELS_FIXED_2_###.png'), 'decision': 'pending_visual_review', 'recommendation': 'Revisar figuras E8_alkafri_FINAL_LABELS_FIXED_2_*.png. Si al menos 30 overlays finales son anatómicamente coherentes, crear subset axial curado y pasar a baseline axial.'}
report_path = E8_ROOT / 'E8_FINAL_LABELS_FIXED_2_report.json'
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
(DOCS_ROOT / 'E8_FINAL_LABELS_FIXED_2_conclusion.md').write_text('# E8 Final labels fixed_2\n\n' + json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(report, indent=2, ensure_ascii=False))
print('Reporte:', report_path)
